# OER Lakehouse — Compute Embeddings

Tính embedding vectors cho toàn bộ tài liệu OER bằng `distiluse-base-multilingual-cased-v2`.

**Trước khi chạy:**
1. `Runtime` → `Change runtime type` → chọn **T4 GPU** (nhanh hơn ~5x, miễn phí)
2. Upload file `oer_texts_for_embedding.jsonl` từ `data/exports/` trên máy của bạn
3. Chạy tất cả cells theo thứ tự
4. Download `oer_embeddings.jsonl` về máy → đặt vào `data/exports/`
5. Chạy: `python scripts/import_embeddings.py data/exports/oer_embeddings.jsonl`

In [ ]:
# Cell 1: Install
!pip install -q sentence-transformers

In [ ]:
# Cell 2: Upload file oer_texts_for_embedding.jsonl từ máy của bạn
from google.colab import files
import json, os

uploaded = files.upload()   # Chọn file oer_texts_for_embedding.jsonl
INPUT_FILE = list(uploaded.keys())[0]
OUTPUT_FILE = "oer_embeddings.jsonl"

line_count = sum(1 for line in open(INPUT_FILE, encoding='utf-8') if line.strip())
print(f"✅ Đã upload: {INPUT_FILE} — {line_count:,} records")

In [ ]:
# Cell 3: Load model
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "distiluse-base-multilingual-cased-v2"
print(f"Loading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
print(f"✅ Model ready — dims={model.get_sentence_embedding_dimension()}")

In [ ]:
# Cell 4: Đọc texts và encode embeddings
BATCH_SIZE = 256

records = []
all_doc_texts = []
all_chunk_texts = []
chunk_map = []  # (record_idx, chunk_idx_in_record)

print("Reading texts...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        rec_idx = len(records)
        records.append(rec)
        all_doc_texts.append(rec.get("doc_text", ""))
        for ch in rec.get("chunks", []):
            all_chunk_texts.append(ch["text"])
            chunk_map.append((rec_idx, ch["idx"]))

print(f"Collected: {len(all_doc_texts):,} docs, {len(all_chunk_texts):,} chunks")

# Encode doc-level
print(f"\nEncoding {len(all_doc_texts):,} doc embeddings...")
doc_embeddings = model.encode(
    all_doc_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"✅ Doc embeddings: {doc_embeddings.shape}")

# Encode chunk-level
chunk_embeddings_all = None
if all_chunk_texts:
    print(f"\nEncoding {len(all_chunk_texts):,} chunk embeddings...")
    chunk_embeddings_all = model.encode(
        all_chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"✅ Chunk embeddings: {chunk_embeddings_all.shape}")

In [ ]:
# Cell 5: Lưu output file
print(f"Writing embeddings to {OUTPUT_FILE} ...")

chunk_embs_by_record = {}
if chunk_embeddings_all is not None:
    for i, (rec_idx, chunk_idx) in enumerate(chunk_map):
        if rec_idx not in chunk_embs_by_record:
            chunk_embs_by_record[rec_idx] = {}
        chunk_embs_by_record[rec_idx][chunk_idx] = chunk_embeddings_all[i].tolist()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for i, rec in enumerate(records):
        chunk_embs_dict = chunk_embs_by_record.get(i, {})
        num_chunks = len(rec.get("chunks", []))
        chunk_embs = [chunk_embs_dict[ci] for ci in range(num_chunks) if ci in chunk_embs_dict]
        output = {
            "_id": rec["_id"],
            "embedding": doc_embeddings[i].tolist(),
            "chunk_embeddings": chunk_embs,
        }
        f.write(json.dumps(output) + "\n")

size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"\n✅ Done! {OUTPUT_FILE} ({size_mb:.1f} MB)")
print(f"   {len(records):,} doc embeddings")
print(f"   {len(all_chunk_texts):,} chunk embeddings")

In [ ]:
# Cell 6: Download file về máy
from google.colab import files
files.download(OUTPUT_FILE)
print("\n📥 Sau khi download:")
print("  1. Đặt file vào: data/exports/oer_embeddings.jsonl")
print("  2. Chạy: python scripts/import_embeddings.py data/exports/oer_embeddings.jsonl")